# Getting Started: Gemma 4 Developer Agent

In [ ]:
# Ten-minute arm: identical to wfl-gemma31b-129-tasks.ipynb except for this cell.
# On Kaggle it ran as a one-line compressed exec of exactly this code (Version 3 of fork-of-new-book).
# max_time_minutes 6 -> 10, the 27 sandbox-broken fastapi tasks are skipped, no new task starts after a fixed UTC deadline.
import shutil,pathlib,time
_ct=shutil.copytree
DROP={'fastapi_14356', 'fastapi_14605', 'fastapi_14246', 'fastapi_14485', 'fastapi_14361', 'fastapi_13713', 'fastapi_14609', 'fastapi_15030', 'fastapi_14962', 'fastapi_14349', 'fastapi_14371', 'fastapi_14482', 'fastapi_14455', 'fastapi_14303', 'fastapi_15745', 'fastapi_14583', 'fastapi_13786', 'fastapi_14459', 'fastapi_14978', 'fastapi_14964', 'fastapi_15023', 'fastapi_14360', 'fastapi_14266', 'fastapi_14953', 'fastapi_14791', 'fastapi_12942', 'fastapi_14512'}
DEADLINE=1794512400
def _p(src,dst,*a,**k):
    r=_ct(src,dst,*a,**k)
    if pathlib.Path(dst).name=='sample_submission' and not getattr(_p,'done',False):
        _p.done=True
        import swegemma.models as M, swegemma.config as C, swegemma.evaluate as E
        _lt=M.load_tasks
        def lt(*a,**k):
            ts=_lt(*a,**k); out=[t for t in ts if t.instance_id not in DROP]
            print('TEN-MINUTE ARM: kept',len(out),'of',len(ts),'tasks'); return out
        M.load_tasks=lt; get_ipython().user_ns['load_tasks']=lt
        _ci=C.EvalConfig.__init__
        def ci(self,*a,**k):
            if 'max_time_minutes' in k: k['max_time_minutes']=10.0
            _ci(self,*a,**k)
            print('TEN-MINUTE ARM: max_time_minutes =',getattr(self,'max_time_minutes',None))
        C.EvalConfig.__init__=ci
        _ev=E.Evaluator.evaluate_task
        async def ev(self,*a,**k):
            if time.time()>DEADLINE: raise RuntimeError('session deadline')
            return await _ev(self,*a,**k)
        E.Evaluator.evaluate_task=ev
        print('TEN-MINUTE ARM armed')
    return r
shutil.copytree=_p
print('ten-minute arm loaded')


## 1. Environment Configuration and Package Installation

In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

## 2. Competition Dataset and Sample Submission

In [ ]:
import re
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Official quantized competition models:
#   - 'gemma-4-31b-it-qat-w4a16-ct' (default in sample_submission)
#   - 'gemma-4-12b-it-qat-w4a16-ct'
#   - 'gemma-4-e4b-it-qat-w4a16-ct'
#   - 'gemma-4-e2b-it-qat-w4a16-ct'
MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'  # research run: the 31B model the paper studies

# Copy sample_submission to a writable directory for customization and packaging
SAMPLE_SUBMISSION_SRC = DATA_DIR / 'sample_submission'
AGENT_DIR = WORKING_DIR / 'sample_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(SAMPLE_SUBMISSION_SRC, AGENT_DIR)

# If switching away from 31B, update all agent YAMLs and remove the 31B-shaped sample LoRA adapters
# Research run: always strip the sample's DUMMY LoRA adapters so we measure the base model
if True:
    for yaml_path in [AGENT_DIR / 'agent.yaml', *sorted((AGENT_DIR / 'sub_agents').glob('*.yaml'))]:
        if yaml_path.is_file():
            text = yaml_path.read_text(encoding='utf-8')
            text = re.sub(r'^(\s*model:\s*).*$', rf'\g<1>{MODEL_NAME}', text, flags=re.MULTILINE)
            text = re.sub(r'^\s*adapter:\s*.*\n?', '', text, flags=re.MULTILINE)
            yaml_path.write_text(text, encoding='utf-8')
    shutil.rmtree(AGENT_DIR / 'adapters', ignore_errors=True)

# Load tasks from the published dataset
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR} (model={MODEL_NAME})')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')
for t in tasks[:5]:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

## 3. Code Graph Functions

In [ ]:
from swegemma import graph as sg

sample_task = tasks[0]
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

# Load the pre-computed repository code graph and node embeddings for the sample task
repo_graph = sg.get_graph(
    repo_name=sample_task.repo,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
sample_nodes = list(repo_graph.nodes())
print(f'Graph for {sample_task.repo} ({sample_task.instance_id}):')
print(f'  Nodes: {repo_graph.number_of_nodes()}, Edges: {repo_graph.number_of_edges()}')

# Select a connected symbol and query structural neighbors (callers, callees, definitions)
query_node = next((n for n, deg in repo_graph.degree() if deg >= 5), sample_nodes[0])
neighbors = sg.get_neighbor(
    node=query_node,
    graph=repo_graph,
    max_neighbors=10,
)
print(f'\nNeighbors of {query_node!r} (showing up to 5):')
for n in neighbors[:5]:
    print(f'  - {n}')

# Search for semantically similar code nodes using pre-computed vector embeddings
similar = sg.get_similar_nodes(
    node=query_node,
    repo_name=sample_task.repo,
    k=5,
    graph=repo_graph,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
print(f'\nTop similar nodes to {query_node!r}:')
for item in similar:
    print(f"  - {item['node_name']} (similarity={item['similarity']:.4f})")

# Extract an induced subgraph connecting the query symbol and its neighbors
focal_nodes = [query_node, *neighbors[:4]]
subgraph = sg.get_induced_subgraph(repo_graph, focal_nodes)
print(
    f'\nInduced subgraph over {len(focal_nodes)} focal nodes: '
    f'{subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges'
)

## 4. Start vLLM Server

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

# Validate single canonical base model and discover any PEFT LoRA adapters in the submission directory.
# Note: sample_submission includes dummy LoRA adapters (adapters/main_lora and adapters/tool_lora)
# shaped for gemma-4-31b-it-qat-w4a16-ct; Cell 2 removes them automatically when MODEL_NAME != 31B.
declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{declared_model}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}. Ensure the model input is attached.'

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), (
        f'Unsafe adapter name {adapter_name!r}: must match [A-Za-z0-9][A-Za-z0-9._-]*'
    )
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),
    max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(detected_ranks) if detected_ranks else 128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

## 5. Full run: Gemma 4 31B on all 129 public tasks (per-task results for the paper)

In [ ]:
# ===== "Where the Fix Lives" — Gemma 4 31B run on all public tasks =====
# Writes one JSON line per finished task to /kaggle/working/gemma31b_task_results.jsonl
# (crash-safe: re-running this cell skips tasks already recorded).
import asyncio, concurrent.futures, json, random, time, traceback
import pandas as pd
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

# ---- research budgets (NOT written into the submission's eval_config.yaml) ----
TIME_PER_TASK_MIN = 6.0      # agent wall-clock per task
TOOL_CALLS_PER_TASK = 40
MAX_TURNS = 70
CONCURRENCY = 2              # tasks in flight at once; set to 1 if you see errors
SESSION_BUDGET_HOURS = 10.5  # stop starting new tasks after this (Kaggle GPU sessions end at 12h)
OUT = WORKING_DIR / 'gemma31b_task_results.jsonl'

limits, gen_constraints = build_submission_limits()
eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=WORKING_DIR / 'results',
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=300,
    verification_timeout_seconds=300,
    max_time_minutes=TIME_PER_TASK_MIN,
    max_tool_calls=TOOL_CALLS_PER_TASK,
    max_turns=MAX_TURNS,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)
evaluator = Evaluator(eval_config)

done = set()
if OUT.exists():
    done = {json.loads(l)['instance_id'] for l in OUT.read_text().splitlines() if l.strip()}
order = sorted(tasks, key=lambda t: t.instance_id)
random.Random(2026).shuffle(order)          # fixed random order: a partial run is still an unbiased sample
todo = [t for t in order if t.instance_id not in done]
print(f'{len(done)} already done, {len(todo)} to run, concurrency={CONCURRENCY}')

T0 = time.time()
lock = asyncio.Lock()

async def one(task, idx, sem):
    async with sem:
        if time.time() - T0 > SESSION_BUDGET_HOURS * 3600:
            return
        t1 = time.time()
        rec = {'instance_id': task.instance_id, 'repo': task.repo}
        try:
            r = await evaluator.evaluate_task(task=task, task_index=idx, total_tasks=len(todo))
            patch = (r.agent_patch or '')
            rec.update(resolved=bool(r.resolved), test_exit_code=r.test_exit_code,
                       tool_calls=r.tool_calls, duration_s=round(r.duration_seconds, 1),
                       patch_chars=len(patch.strip()),
                       patch_files=sorted({l.split(' b/')[-1] for l in patch.splitlines() if l.startswith('diff --git')}),
                       error=str(getattr(r, 'error', '') or '')[:500])
        except Exception as e:
            rec.update(resolved=False, error=f'{type(e).__name__}: {e}'[:500],
                       duration_s=round(time.time() - t1, 1))
            traceback.print_exc()
        async with lock:
            with open(OUT, 'a') as fh:
                fh.write(json.dumps(rec) + '\n')
            n = sum(1 for _ in open(OUT))
            print(f"[{n}/{len(tasks)}] {task.instance_id}: resolved={rec['resolved']} "
                  f"calls={rec.get('tool_calls')} t={rec['duration_s']}s  elapsed={(time.time()-T0)/3600:.2f}h", flush=True)

async def main():
    sem = asyncio.Semaphore(CONCURRENCY)
    await asyncio.gather(*(one(t, i, sem) for i, t in enumerate(todo, 1)))

def run_in_thread(coro_fn):
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(lambda: asyncio.run(coro_fn())).result()

run_in_thread(main)

df = pd.read_json(OUT, lines=True)
print(f"\nDONE: {df.resolved.sum()}/{len(df)} resolved ({df.resolved.mean():.1%}) in {(time.time()-T0)/3600:.2f}h")
print(df.groupby('repo').resolved.agg(['sum', 'count']))
df.to_csv(WORKING_DIR / 'gemma31b_task_results.csv', index=False)
